# Phân tích trường `LogonID` & `SubjectLogonID` — Ngày 16

Notebook kiểm tra toàn diện trường `LogonID` và đối chiếu với `SubjectLogonID` trên cả **Event 4624** (Successful Logon) và **Event 4625** (Failed Logon).

### Bối cảnh kỹ thuật & Nghiệp vụ Windows Security:
- **`LogonID` (LUID - Locally Unique Identifier):** Là mã định danh phiên đăng nhập 64-bit (thường biểu diễn dưới dạng số thập lục phân như `0x3e7`, `0x1a3f1444`).
- **Event 4624 (Thành công):** Windows cấp một phiên làm việc mới cho tài khoản đăng nhập thành công. Do đó, trường này có **Cardinality cực lớn (>16.9 triệu giá trị duy nhất)** và tỷ lệ thiếu gần như bằng 0 (chỉ 0.16%).
- **Event 4625 (Thất bại):** Khi đăng nhập bị từ chối, Windows không cấp session mới. `LogonID` bị thiếu **49.16%** (toàn bộ ở Network Logon). Khi có mặt, **>90% bản ghi mang giá trị cố định `0x3e7`** (`LOCAL_SYSTEM`).
- **`SubjectLogonID`:** Mã phiên của chủ thể yêu cầu đăng nhập. Trường này bị thiếu **99.92%** ở 4624 và **100%** ở 4625, chỉ xuất hiện duy nhất ở kiểu đăng nhập `NewCredentials` (LogonType 9).

Notebook được thiết kế bằng **PyArrow Streaming & C++ Compute Kernels** để xử lý 17.9 triệu dòng chỉ trong vài giây, RAM luôn duy trì < 500MB.

In [1]:
from collections import Counter
from pathlib import Path
import sys
from typing import Any, Dict, List, Optional, Tuple

# Đảm bảo hiển thị an toàn UTF-8 trên console Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

# Cấu hình tham số - dễ dàng mở rộng và tái sử dụng cho các ngày hoặc cột khác
DAY_TAG = "day-16"
BATCH_SIZE = 250_000
TOP_N = 15

# Các mã phiên đặc quyền chuẩn của hệ thống Windows (Well-known System LUIDs)
SYSTEM_LUIDS = {
    "0x3e7": "Local System (SYSTEM)",
    "0x3e5": "Local Service",
    "0x3e4": "Network Service",
}

PARQUET_RELATIVE_PATHS = {
    "Event 4624": Path(f"data/interim/event_4624/event_4624_{DAY_TAG}.parquet"),
    "Event 4625": Path(f"data/interim/event_4625/event_4625_{DAY_TAG}.parquet"),
}


def find_project_file(relative_path: Path, start_directory: Optional[Path] = None) -> Path:
    """Tìm file từ thư mục hiện tại hoặc một trong các thư mục cha."""
    start_directory = (start_directory or Path.cwd()).resolve()

    for directory in (start_directory, *start_directory.parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy '{relative_path}' từ '{start_directory}' hoặc các thư mục cha."
    )


parquet_paths = {
    event_name: find_project_file(relative_path)
    for event_name, relative_path in PARQUET_RELATIVE_PATHS.items()
}

for event_name, parquet_path in parquet_paths.items():
    print(f"{event_name}: {parquet_path}")


Event 4624: D:\Github Repo\ueba-benchmark\data\interim\event_4624\event_4624_day-16.parquet
Event 4625: D:\Github Repo\ueba-benchmark\data\interim\event_4625\event_4625_day-16.parquet


## 1. Hàm tiện ích kiểm tra giá trị thiếu và tính toán thống kê

Sử dụng `pyarrow.compute` để kiểm tra chuỗi thiếu (`null`, rỗng `""`, hoặc toàn khoảng trắng) trực tiếp ở cấp độ C++.

In [2]:
def missing_text_mask(values: pa.Array) -> pa.Array:
    """Trả về mask boolean cho null, chuỗi rỗng và chuỗi chỉ có khoảng trắng."""
    values_without_null = pc.fill_null(values, "")
    return pc.equal(pc.utf8_trim_whitespace(values_without_null), "")


def compute_id_column_stats(file_path: Path, column: str) -> Dict[str, Any]:
    """Tính số lượng tổng, missing, valid và cardinality bằng PyArrow compute."""
    table = pq.read_table(file_path, columns=[column])
    col_data = table[column]
    total = len(col_data)
    mask = missing_text_mask(col_data)
    missing_cnt = pc.sum(mask.cast(pa.int64())).as_py()
    valid_cnt = total - missing_cnt
    missing_rate = (missing_cnt / total * 100) if total > 0 else 0.0
    
    # Tính số lượng giá trị duy nhất (Cardinality) trên tập hợp hợp lệ
    valid_data = pc.filter(col_data, pc.invert(mask))
    unique_cnt = pc.count_distinct(valid_data, mode="only_valid").as_py()
    
    return {
        "Column": column,
        "TotalRows": total,
        "MissingRows": missing_cnt,
        "ValidRows": valid_cnt,
        "MissingRatePct": round(missing_rate, 2),
        "UniqueCount": unique_cnt,
    }


## 2. Thống kê tổng quan Tính Đầy Đủ & Cardinality của `LogonID` và `SubjectLogonID`

Bảng dưới đây so sánh hai trường định danh phiên giữa Event 4624 và Event 4625.

In [3]:
overview_rows = []
for event_name, path in parquet_paths.items():
    for col in ["LogonID", "SubjectLogonID"]:
        stats = compute_id_column_stats(path, col)
        stats["Event"] = event_name
        overview_rows.append(stats)

df_overview = pd.DataFrame(overview_rows)[
    ["Event", "Column", "TotalRows", "MissingRows", "ValidRows", "MissingRatePct", "UniqueCount"]
]
display(df_overview)


,Event,Column,TotalRows,MissingRows,ValidRows,MissingRatePct,UniqueCount
0,Event 4624,LogonID,17553517,28153,17525364,0.16,16931276
1,Event 4624,SubjectLogonID,17553517,17540066,13451,99.92,58
2,Event 4625,LogonID,359897,176924,182973,49.16,81
3,Event 4625,SubjectLogonID,359897,359897,0,100.00,0


## 3. Phân rã nguyên nhân thiếu `LogonID` theo `LogonType`

Khảo sát mối liên hệ giữa kiểu đăng nhập (`LogonType`) và việc thiếu `LogonID`.

In [4]:
def compute_logon_id_by_logon_type(file_path: Path, batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    """Thống kê số lượng missing LogonID theo cặp (LogonType, LogonTypeDescription)."""
    parquet_file = pq.ParquetFile(file_path)
    counts: Dict[Tuple[Any, Any], Dict[str, int]] = {}

    for batch in parquet_file.iter_batches(
        columns=["LogonType", "LogonTypeDescription", "LogonID"],
        batch_size=batch_size
    ):
        df_b = batch.to_pandas()
        is_missing = df_b["LogonID"].isna() | df_b["LogonID"].astype(str).str.strip().eq("")
        grouped = df_b.groupby(["LogonType", "LogonTypeDescription"], dropna=False).size()
        miss_grouped = df_b[is_missing].groupby(["LogonType", "LogonTypeDescription"], dropna=False).size()

        for key, total_cnt in grouped.items():
            if key not in counts:
                counts[key] = {"Total": 0, "Missing": 0}
            counts[key]["Total"] += int(total_cnt)
            counts[key]["Missing"] += int(miss_grouped.get(key, 0))

    rows = []
    for (ltype, ldesc), val in counts.items():
        total = val["Total"]
        missing = val["Missing"]
        valid = total - missing
        rate = round(missing / total * 100, 2) if total > 0 else 0.0
        rows.append({
            "LogonType": ltype,
            "LogonTypeDescription": ldesc,
            "Total": total,
            "Missing": missing,
            "Valid": valid,
            "MissingRatePct": rate,
        })

    return pd.DataFrame(rows).sort_values(by="Total", ascending=False).reset_index(drop=True)


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Phân rã missing LogonID theo LogonType\n{'=' * 80}")
    df_ltype = compute_logon_id_by_logon_type(path)
    display(df_ltype)



Event 4624: Phân rã missing LogonID theo LogonType


,LogonType,LogonTypeDescription,Total,Missing,Valid,MissingRatePct
0,3,Network,16794417,28153,16766264,0.17
1,5,Service,373734,0,373734,0.00
2,2,Interactive,219613,0,219613,0.00
3,7,Unlock,78760,0,78760,0.00
4,8,NetworkClearText,62286,0,62286,0.00
5,9,NewCredentials,13451,0,13451,0.00
6,0,Local System,5786,0,5786,0.00
7,4,Batch,2321,0,2321,0.00
8,10,RemoteInteractive,2230,0,2230,0.00
9,11,CachedInteractive,916,0,916,0.00



Event 4625: Phân rã missing LogonID theo LogonType


,LogonType,LogonTypeDescription,Total,Missing,Valid,MissingRatePct
0,3,Network,334377,176924,157453,52.91
1,2,Interactive,18872,0,18872,0.00
2,7,Unlock,3624,0,3624,0.00
3,4,Batch,1906,0,1906,0.00
4,11,CachedInteractive,588,0,588,0.00
5,10,RemoteInteractive,267,0,267,0.00
6,5,Service,146,0,146,0.00
7,8,NetworkClearText,117,0,117,0.00


## 4. Phân tích các Well-known System LUIDs (`0x3e7`, `0x3e5`, `0x3e4`)

Trong Windows, các tiến trình và dịch vụ nền đặc quyền sử dụng các mã LUID cố định:
- `0x3e7` (decimal 999): `LOCAL_SYSTEM`
- `0x3e5` (decimal 997): `LOCAL_SERVICE`
- `0x3e4` (decimal 996): `NETWORK_SERVICE`

Bảng dưới đây thống kê số lượng bản ghi gắn với các phiên hệ thống này so với các phiên của người dùng thông thường.

In [5]:
def analyze_system_luids(file_path: Path) -> pd.DataFrame:
    """Thống kê tỷ lệ các phiên hệ thống đặc quyền."""
    table = pq.read_table(file_path, columns=["LogonID"])
    vc = pc.value_counts(table["LogonID"])
    val_dict = dict(zip(vc.field("values").to_pylist(), vc.field("counts").to_pylist()))

    total_valid = sum(v for k, v in val_dict.items() if k is not None and str(k).strip() != "")
    rows = []
    system_total = 0

    for luid, desc in SYSTEM_LUIDS.items():
        cnt = val_dict.get(luid, 0)
        system_total += cnt
        rows.append({
            "Category": f"System: {desc} ({luid})",
            "Count": cnt,
            "PercentageOfValid": round(cnt / total_valid * 100, 2) if total_valid > 0 else 0.0,
        })

    user_sessions_cnt = total_valid - system_total
    rows.append({
        "Category": "User / Ephemeral Sessions (Khác)",
        "Count": user_sessions_cnt,
        "PercentageOfValid": round(user_sessions_cnt / total_valid * 100, 2) if total_valid > 0 else 0.0,
    })

    return pd.DataFrame(rows)


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Tỷ trọng System LUIDs vs User Sessions\n{'=' * 80}")
    df_sys = analyze_system_luids(path)
    display(df_sys)



Event 4624: Tỷ trọng System LUIDs vs User Sessions


,Category,Count,PercentageOfValid
0,System: Local System (SYSTEM) (0x3e7),367264,2.10
1,System: Local Service (0x3e5),5820,0.03
2,System: Network Service (0x3e4),5819,0.03
3,User / Ephemeral Sessions (Khác),17146461,97.84



Event 4625: Tỷ trọng System LUIDs vs User Sessions


,Category,Count,PercentageOfValid
0,System: Local System (SYSTEM) (0x3e7),165271,90.33
1,System: Local Service (0x3e5),0,0.00
2,System: Network Service (0x3e4),59,0.03
3,User / Ephemeral Sessions (Khác),17643,9.64


## 5. Tần suất tái sử dụng phiên (Session Reuse / Long-lived Sessions)

Hầu hết các phiên đăng nhập thông thường chỉ phát sinh 1 hoặc vài event. Tuy nhiên, có những `LogonID` xuất hiện hàng chục nghìn lần trong ngày.
Đây là dấu hiệu của các **Service Accounts** hoặc các tiến trình chạy ngầm liên tục (Persistent Sessions) — tín hiệu quan trọng trong phân tích baseline UEBA.

In [6]:
def get_top_reused_logon_ids(file_path: Path, top_n: int = TOP_N) -> pd.DataFrame:
    """Tìm top các LogonID xuất hiện nhiều lần nhất kèm ngữ cảnh User và Host."""
    table = pq.read_table(file_path, columns=["LogonID", "UserName", "LogHost", "LogonTypeDescription"])
    vc = pc.value_counts(table["LogonID"])
    struct_table = pa.Table.from_arrays([vc.field("values"), vc.field("counts")], names=["LogonID", "Count"])
    
    # Lọc bỏ null
    mask_valid = pc.is_valid(struct_table["LogonID"])
    struct_table = struct_table.filter(mask_valid)
    
    sorted_indices = pc.sort_indices(struct_table["Count"], sort_keys=[("Count", "descending")])
    top_struct = struct_table.take(sorted_indices.slice(0, top_n)).to_pandas()
    
    # Lấy thông tin đại diện (UserName, LogHost, LogonType) cho từng top LogonID
    context_rows = []
    for _, row in top_struct.iterrows():
        lid = row["LogonID"]
        cnt = row["Count"]
        mask_lid = pc.equal(table["LogonID"], lid)
        sample = table.filter(mask_lid).slice(0, 1).to_pandas().iloc[0]
        
        context_rows.append({
            "LogonID": lid,
            "EventCount": cnt,
            "UserName": sample["UserName"],
            "LogHost": sample["LogHost"],
            "LogonTypeDescription": sample["LogonTypeDescription"],
            "IsSystemLUID": lid in SYSTEM_LUIDS,
        })
        
    return pd.DataFrame(context_rows)


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Top {TOP_N} LogonID xuất hiện nhiều lần nhất\n{'=' * 80}")
    df_top_lids = get_top_reused_logon_ids(path, top_n=TOP_N)
    display(df_top_lids)



Event 4624: Top 15 LogonID xuất hiện nhiều lần nhất


,LogonID,EventCount,UserName,LogHost,LogonTypeDescription,IsSystemLUID
0,0x3e7,367264,system,Comp995416,Service,True
1,0x30c896b7,20102,Administrator,Comp415540,Service,False
2,0x41227eb2,19221,Administrator,Comp044849,Service,False
3,0x1a3f1444,11331,Administrator,Comp297849,Service,False
4,0x18d75473,7692,Administrator,Comp297849,Service,False
5,0x280169da,6797,Administrator,Comp415540,Network,False
6,0x3b21d49f,6625,Administrator,Comp044849,Network,False
7,0x13db98ae,6457,Administrator,Comp297849,Network,False
8,0x3e5,5820,local service,EnterpriseAppServer,Service,True
9,0x3e4,5819,network service,EnterpriseAppServer,Service,True



Event 4625: Top 15 LogonID xuất hiện nhiều lần nhất


,LogonID,EventCount,UserName,LogHost,LogonTypeDescription,IsSystemLUID
0,0x3e7,165271,User668517,EnterpriseAppServer,Batch,True
1,0x1fe11,9071,AppService,EnterpriseAppServer,Interactive,False
2,0x1e0fb,8004,AppService,EnterpriseAppServer,Interactive,False
3,0x23c52,78,User857646,EnterpriseAppServer,Network,False
4,0x10e6fe5c,60,User297403,Comp317953,Batch,False
5,0x248cc28,60,User297403,Comp186428,Batch,False
6,0x3e4,59,EnterpriseAppServer$,EnterpriseAppServer,Network,True
7,0xaf6c26,46,User775940,Comp029382,Network,False
8,0x2a7ce,29,User857646,EnterpriseAppServer,Network,False
9,0x378f233e,25,User637832,Comp186428,Network,False


## 6. Khảo sát đặc thù trường `SubjectLogonID` (LogonType 9 - `NewCredentials`)

Trong Windows Security, `SubjectLogonID` là ID phiên của tiến trình gọi yêu cầu đăng nhập.
Dưới đây là bằng chứng thực nghiệm cho thấy `SubjectLogonID` xuất hiện khi nào và mối quan hệ giữa `SubjectLogonID` (Caller Session) và `LogonID` (Target Session).

In [7]:
def inspect_subject_logon_id(file_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Khảo sát các bản ghi có SubjectLogonID hợp lệ."""
    cols = ["LogonType", "LogonTypeDescription", "SubjectLogonID", "LogonID", "SubjectUserName", "UserName", "ProcessName"]
    table = pq.read_table(file_path, columns=cols)
    mask = pc.invert(missing_text_mask(table["SubjectLogonID"]))
    filtered = table.filter(mask)
    
    if len(filtered) == 0:
        return pd.DataFrame(), pd.DataFrame()
        
    df_valid = filtered.to_pandas()
    summary = df_valid.groupby(["LogonType", "LogonTypeDescription"], dropna=False).size().reset_index(name="Count")
    sample = df_valid.head(10)
    return summary, sample


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Khảo sát SubjectLogonID\n{'=' * 80}")
    df_sum, df_sample = inspect_subject_logon_id(path)
    if df_sum.empty:
        print("Không có bản ghi nào chứa SubjectLogonID hợp lệ (100% missing).")
    else:
        print(f"Tìm thấy {len(df_sample):,} dòng mẫu trên tổng số bản ghi có SubjectLogonID:")
        display(df_sum)
        display(df_sample)



Event 4624: Khảo sát SubjectLogonID
Tìm thấy 10 dòng mẫu trên tổng số bản ghi có SubjectLogonID:


,LogonType,LogonTypeDescription,Count
0,9,NewCredentials,13451


,LogonType,LogonTypeDescription,SubjectLogonID,LogonID,SubjectUserName,UserName,ProcessName
0,9,NewCredentials,0x3e7,0x11d1da25,Comp434746$,system,svchost.exe
1,9,NewCredentials,0x2e567,0xe81c37b9,User171300,User171300,Proc777795.exe
2,9,NewCredentials,0x2e567,0xe81cc211,User171300,User171300,Proc777795.exe
3,9,NewCredentials,0x2e567,0xe81cd5b0,User171300,User171300,Proc777795.exe
4,9,NewCredentials,0x29e93,0x5af26e62,User793535,User793535,Proc964159.exe
5,9,NewCredentials,0x29e93,0x5af2c77b,User793535,User793535,Proc964159.exe
6,9,NewCredentials,0x2e567,0xe81d5577,User171300,User171300,Proc777795.exe
7,9,NewCredentials,0x29e93,0x5af365b9,User793535,User793535,Proc964159.exe
8,9,NewCredentials,0x2e567,0xe81dd7df,User171300,User171300,Proc777795.exe
9,9,NewCredentials,0x2e567,0xe81df968,User171300,User171300,Proc777795.exe



Event 4625: Khảo sát SubjectLogonID
Không có bản ghi nào chứa SubjectLogonID hợp lệ (100% missing).


## 7. Kết luận và Khuyến nghị cho UEBA Benchmark

1. **Bản chất của `LogonID` trong Feature Engineering:**
   - **Không bao giờ dùng raw string `LogonID` làm input cho mô hình Machine Learning** (vì Cardinality lên tới 16.9 triệu giá trị, là chuỗi LUID ngẫu nhiên sinh ra trong runtime của OS).
   - Các biến trích xuất hữu ích cho UEBA:
     - `IsSystemLogonID`: Cờ nhị phân (1 nếu `LogonID in ['0x3e7', '0x3e5', '0x3e4']`, 0 nếu khác).
     - `LogonID_SessionVolume`: Đếm số lượng event phát sinh trong cùng một `LogonID` để nhận diện các phiên chạy ngầm dài hạn (Long-lived Session) vs phiên 1 lần (Ephemeral).
     - `HasSubjectLogonID`: Đánh dấu các phiên nâng quyền / đổi ngữ cảnh credential (`runas` / `NewCredentials`).

2. **Ý nghĩa an ninh mạng của `LogonID`:**
   - Trong điều tra số (Digital Forensics) và xâu chuỗi sự kiện (Session Tracking): `LogonID` cho phép nhóm các hành vi của cùng một phiên đăng nhập (từ logon thành công 4624 đến khi logoff 4634 hoặc thực thi tiến trình 4688).
   - Ở Event 4625 (thất bại): Việc `0x3e7` chiếm đa số chứng minh các cuộc xác thực thất bại thường do tiến trình hệ thống (LSASS/RPC/Service) tiếp nhận và báo lỗi.